In [1]:
import numpy as np
import pandas as pd

#load the dataset
df = pd.read_csv('train_test_network.csv')
print("Dataset shape:" , df.shape)
df.head()

Dataset shape: (211043, 44)


,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [2]:
#show number of benign and malicious samples
print(df['label'].value_counts())
#show number of attack types
print(df['type'].value_counts())

label
1    161043
0     50000
Name: count, dtype: int64
type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
scanning      20000
ransomware    20000
xss           20000
mitm           1043
Name: count, dtype: int64


In [3]:
#show type for each column
print(df.dtypes)

src_ip                     object
src_port                    int64
dst_ip                     object
dst_port                    int64
proto                      object
service                    object
duration                  float64
src_bytes                   int64
dst_bytes                   int64
conn_state                 object
missed_bytes                int64
src_pkts                    int64
src_ip_bytes                int64
dst_pkts                    int64
dst_ip_bytes                int64
dns_query                  object
dns_qclass                  int64
dns_qtype                   int64
dns_rcode                   int64
dns_AA                     object
dns_RD                     object
dns_RA                     object
dns_rejected               object
ssl_version                object
ssl_cipher                 object
ssl_resumed                object
ssl_established            object
ssl_subject                object
ssl_issuer                 object
http_trans_dep

In [4]:
#replace null values with na
df.replace("-", "n/a", inplace=True)
df.head()

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,n/a,290.371539,101568,2592,OTH,...,0,0,n/a,n/a,n/a,n/a,n/a,n/a,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,n/a,0.000102,0,0,REJ,...,0,0,n/a,n/a,n/a,n/a,n/a,n/a,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,n/a,0.000148,0,0,REJ,...,0,0,n/a,n/a,n/a,n/a,n/a,n/a,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,n/a,0.000113,0,0,REJ,...,0,0,n/a,n/a,n/a,n/a,n/a,n/a,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,n/a,0.000130,0,0,REJ,...,0,0,n/a,n/a,n/a,n/a,n/a,n/a,1,backdoor


In [5]:
#remove duplicate rows
df.drop_duplicates(inplace=True)
print("Dataset shape after removing duplicates:" , df.shape)


Dataset shape after removing duplicates: (190474, 44)


In [6]:
#apply one-hot encoding or binary encoding on selected categorical columns
one_hot_columns = ["proto", "service", "conn_state",
                   "dns_AA", "dns_RD", "dns_RA", "dns_rejected",
                   "ssl_resumed", "ssl_established",
                   "http_trans_depth", "http_method", "http_version",
                   "weird_addl", "weird_notice", 
                   ]

binary_columns = ["dns_query",
                  "ssl_version", "ssl_cipher", "ssl_subject", "ssl_issuer",
                  "http_uri", "http_user_agent", "http_orig_mime_types", "http_resp_mime_types",
                  "weird_name", 
                  ]

#change object columns to category dtype
for col in one_hot_columns + binary_columns:
    df[col] = df[col].astype('category')
    


In [7]:
#apply one-hot encoding
df = pd.get_dummies(df, columns=one_hot_columns, sparse=False)

df.head()

,src_ip,src_port,dst_ip,dst_port,duration,src_bytes,dst_bytes,missed_bytes,src_pkts,src_ip_bytes,...,http_method_HEAD,http_method_POST,http_method_n/a,http_version_1.1,http_version_n/a,weird_addl_46,weird_addl_48,weird_addl_n/a,weird_notice_F,weird_notice_n/a
0,192.168.1.37,4444,192.168.1.193,49178,290.371539,101568,2592,0,108,108064,...,False,False,True,False,True,False,False,True,False,True
1,192.168.1.193,49180,192.168.1.37,8080,0.000102,0,0,0,1,52,...,False,False,True,False,True,False,False,True,False,True
2,192.168.1.193,49180,192.168.1.37,8080,0.000148,0,0,0,1,52,...,False,False,True,False,True,False,False,True,False,True
3,192.168.1.193,49180,192.168.1.37,8080,0.000113,0,0,0,1,48,...,False,False,True,False,True,False,False,True,False,True
4,192.168.1.193,49180,192.168.1.37,8080,0.000130,0,0,0,1,52,...,False,False,True,False,True,False,False,True,False,True


In [8]:
#apply bindary encoding
for col in binary_columns:
    df[col] = (df[col] != 'n/a').astype(int)

df.head()

,src_ip,src_port,dst_ip,dst_port,duration,src_bytes,dst_bytes,missed_bytes,src_pkts,src_ip_bytes,...,http_method_HEAD,http_method_POST,http_method_n/a,http_version_1.1,http_version_n/a,weird_addl_46,weird_addl_48,weird_addl_n/a,weird_notice_F,weird_notice_n/a
0,192.168.1.37,4444,192.168.1.193,49178,290.371539,101568,2592,0,108,108064,...,False,False,True,False,True,False,False,True,False,True
1,192.168.1.193,49180,192.168.1.37,8080,0.000102,0,0,0,1,52,...,False,False,True,False,True,False,False,True,False,True
2,192.168.1.193,49180,192.168.1.37,8080,0.000148,0,0,0,1,52,...,False,False,True,False,True,False,False,True,False,True
3,192.168.1.193,49180,192.168.1.37,8080,0.000113,0,0,0,1,48,...,False,False,True,False,True,False,False,True,False,True
4,192.168.1.193,49180,192.168.1.37,8080,0.000130,0,0,0,1,52,...,False,False,True,False,True,False,False,True,False,True


In [9]:
#feature elimination
removed_columns = ["src_ip", "src_port", "dst_ip", "dst_port"]
df.drop(columns=removed_columns, inplace=True)
print("Dataset shape after feature elimination:" , df.shape)

df.head()

Dataset shape after feature elimination: (190474, 91)


,duration,src_bytes,dst_bytes,missed_bytes,src_pkts,src_ip_bytes,dst_pkts,dst_ip_bytes,dns_query,dns_qclass,...,http_method_HEAD,http_method_POST,http_method_n/a,http_version_1.1,http_version_n/a,weird_addl_46,weird_addl_48,weird_addl_n/a,weird_notice_F,weird_notice_n/a
0,290.371539,101568,2592,0,108,108064,31,3832,0,0,...,False,False,True,False,True,False,False,True,False,True
1,0.000102,0,0,0,1,52,1,40,0,0,...,False,False,True,False,True,False,False,True,False,True
2,0.000148,0,0,0,1,52,1,40,0,0,...,False,False,True,False,True,False,False,True,False,True
3,0.000113,0,0,0,1,48,1,40,0,0,...,False,False,True,False,True,False,False,True,False,True
4,0.000130,0,0,0,1,52,1,40,0,0,...,False,False,True,False,True,False,False,True,False,True


In [10]:
#show type for each column
print(df.dtypes)

duration            float64
src_bytes             int64
dst_bytes             int64
missed_bytes          int64
src_pkts              int64
                     ...   
weird_addl_46          bool
weird_addl_48          bool
weird_addl_n/a         bool
weird_notice_F         bool
weird_notice_n/a       bool
Length: 91, dtype: object


In [11]:
#show class distribution before SMOTE
print("Class distribution before SMOTE:")
print(df['label'].value_counts())


Class distribution before SMOTE:
label
1    148434
0     42040
Name: count, dtype: int64


In [12]:
#apply smote for class balancing
from imblearn.over_sampling import SMOTE
X = df.drop(columns = ['label', 'type'], axis=1)
y = df['label']

print(X.shape)
print(y.shape)
print(df.shape)


(190474, 89)
(190474,)
(190474, 91)


In [13]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
df_resampled = pd.concat([X_resampled, y_resampled], axis=1)


In [14]:
#show class distribution after SMOTE
print("Class distribution after SMOTE:")
print(df_resampled['label'].value_counts())
df_resampled.head()
print(df_resampled.info())

Class distribution after SMOTE:
label
1    148434
0    148434
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296868 entries, 0 to 296867
Data columns (total 90 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   duration                296868 non-null  float64
 1   src_bytes               296868 non-null  int64  
 2   dst_bytes               296868 non-null  int64  
 3   missed_bytes            296868 non-null  int64  
 4   src_pkts                296868 non-null  int64  
 5   src_ip_bytes            296868 non-null  int64  
 6   dst_pkts                296868 non-null  int64  
 7   dst_ip_bytes            296868 non-null  int64  
 8   dns_query               296868 non-null  int64  
 9   dns_qclass              296868 non-null  int64  
 10  dns_qtype               296868 non-null  int64  
 11  dns_rcode               296868 non-null  int64  
 12  ssl_version             296868 non-null 

In [15]:
#save the preprocessed dataset
df_resampled.to_csv('preprocessed_network_dataset.csv', index=False)